<a href="https://colab.research.google.com/github/amann45/AI_Labworks/blob/main/Lab6/lab6_RNN_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab6: RNN [Aman Kumar Ray: ACE080BCT010]
### Objective: To understand the code and find out what the hidden_size means for an RNN Model.


## Theory

### Recurrent Neural Network (RNN)
RNN is a neural network that uses feedback connections to process sequences by passing information from one time step to the next through a hidden state.

### Why Do We Need RNN?
Traditional neural networks treat every input independently.

For example, in the sentence:

"The cat sat on the ___"

To predict the missing word ("mat"), the model needs information from the previous words.

An RNN remembers previous words and uses them to predict the next one.

### Architecture of an RNN

```text
Input x₁ ───▶ [RNN Cell] ───▶ h₁ ───▶ Output y₁
                 │
                 ▼
Input x₂ ───▶ [RNN Cell] ───▶ h₂ ───▶ Output y₂
                 │
                 ▼
Input x₃ ───▶ [RNN Cell] ───▶ h₃ ───▶ Output y₃
                 │
                 ▼
Input x₄ ───▶ [RNN Cell] ───▶ h₄ ───▶ Output y₄
```

### Where:

- **xₜ** = Input at time *t*
- **hₜ** = Hidden state (memory)
- **yₜ** = Output at time *t*

The hidden state carries information from one time step to the next.



### Working of an RNN

At each time step:

1. Receive the current input.
2. Combine it with the previous hidden state.
3. Generate a new hidden state.
4. Produce the output.



### Mathematical Representation

\[
h_t = f(W_{hh}h_{t-1} + W_{xh}x_t + b)
\]

\[
y_t = W_{hy}h_t
\]

### Where:

- \(h_t\) = Current hidden state
- \(h_{t-1}\) = Previous hidden state
- \(x_t\) = Current input
- \(W\) = Weight matrices
- \(b\) = Bias
- \(f\) = Activation function (usually **tanh** or **ReLU**)



### Example

Suppose the input sentence is:

```text
I love deep learning
```

The RNN processes one word at a time.

| Time | Input | Hidden State | Output |
|------|-------|--------------|--------|
| 1 | I | h₁ | y₁ |
| 2 | love | h₂ | y₂ |
| 3 | deep | h₃ | y₃ |
| 4 | learning | h₄ | y₄ |

Each hidden state contains information from all previous words.

In [15]:
import random
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Generate dataset

dishes = ['A', 'B', 'C']
weather_types = ['Sunny', 'Rainy']

dish_to_idx = {d: i for i, d in enumerate(dishes)}
weather_to_idx = {w: i for i, w in enumerate(weather_types)}

def next_dish(dish):
    if dish == 'A':
        return 'B'
    elif dish == 'B':
        return 'C'
    else:
        return 'A'

def generate_sequence(length=1000):
    """
    Returns:
        inputs  = [(dish_t, weather_t), ...]
        targets = [dish_{t+1}, ...]
    """
    current_dish = 'A'

    inputs = []
    targets = []

    for _ in range(length):

        weather = random.choice(weather_types)

        inputs.append((current_dish, weather))

        if weather == 'Sunny':
            new_dish = current_dish
        else:  # Rainy
            new_dish = next_dish(current_dish)

        targets.append(new_dish)

        current_dish = new_dish

    return inputs, targets

In [16]:
# 2. Encode data
def encode_input(dish, weather):
    """
    One-hot encode dish (3 dims)
    +
    One-hot encode weather (2 dims)

    Total input size = 5
    """
    x = torch.zeros(5)

    x[dish_to_idx[dish]] = 1.0
    x[3 + weather_to_idx[weather]] = 1.0

    return x


inputs, targets = generate_sequence(length=2000)

X = torch.stack([
    encode_input(d, w)
    for d, w in inputs
])

y = torch.tensor([
    dish_to_idx[t]
    for t in targets
])

# RNN expects:
# (batch_size, seq_len, input_size)

X = X.unsqueeze(0)      # (1, seq_len, 5)
y = y.unsqueeze(0)      # (1, seq_len)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: torch.Size([1, 2000, 5])
y shape: torch.Size([1, 2000])


In [17]:
# 3. Vanilla RNN model

class DishRNN(nn.Module):
    def __init__(self,
                 input_size=5,
                 hidden_size=3,
                 output_size=3):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
            bias=False,
            nonlinearity = "tanh"
        )

        self.fc = nn.Linear(hidden_size, output_size, bias=False)

    def forward(self, x):
        # rnn_out:
        # (batch, seq_len, hidden_size)

        rnn_out, hidden = self.rnn(x)

        logits = self.fc(rnn_out)

        return logits


model = DishRNN()

In [18]:
model

DishRNN(
  (rnn): RNN(5, 3, bias=False, batch_first=True)
  (fc): Linear(in_features=3, out_features=3, bias=False)
)

In [19]:
# 4. Training setup

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [20]:
# 5. Training loop
epochs = 300

for epoch in range(epochs):

    optimizer.zero_grad()

    logits = model(X)

    # reshape for CE loss
    loss = criterion(
        logits.view(-1, 3),
        y.view(-1)
    )

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss = {loss.item():.6f}"
        )

Epoch [50/300] Loss = 0.693303
Epoch [100/300] Loss = 0.241859
Epoch [150/300] Loss = 0.100535
Epoch [200/300] Loss = 0.057635
Epoch [250/300] Loss = 0.038572
Epoch [300/300] Loss = 0.028084


In [21]:
# 6. Evaluation

with torch.no_grad():

    logits = model(X)

    preds = logits.argmax(dim=-1)

    accuracy = (
        (preds == y).float().mean().item()
    )

print("\nAccuracy:", accuracy)


Accuracy: 1.0


In [22]:
# 7. Test on all possible transitions

print("\nLearned transitions:\n")

test_cases = [
    ('A', 'Sunny'),
    ('A', 'Rainy'),
    ('B', 'Sunny'),
    ('B', 'Rainy'),
    ('C', 'Sunny'),
    ('C', 'Rainy'),
]

hidden = None

for dish, weather in test_cases:

    x = encode_input(dish, weather)
    x = x.unsqueeze(0).unsqueeze(0)

    with torch.no_grad():
        logits = model(x)
        pred = logits.argmax(-1).item()

    print(
        f"Current Dish={dish:1s}, "
        f"Weather={weather:5s} "
        f"--> Predicted Next Dish={dishes[pred]}"
    )


Learned transitions:

Current Dish=A, Weather=Sunny --> Predicted Next Dish=A
Current Dish=A, Weather=Rainy --> Predicted Next Dish=B
Current Dish=B, Weather=Sunny --> Predicted Next Dish=B
Current Dish=B, Weather=Rainy --> Predicted Next Dish=C
Current Dish=C, Weather=Sunny --> Predicted Next Dish=C
Current Dish=C, Weather=Rainy --> Predicted Next Dish=A


In [23]:
for name, param in model.named_parameters():
    print(name)
    print(param.data)
    print()

rnn.weight_ih_l0
tensor([[-0.1469,  2.6165, -2.1740, -1.3818,  1.5315],
        [ 1.8605, -0.0134, -2.2319,  1.4680, -1.4671],
        [ 2.0111, -1.6326,  0.5193, -2.0614,  0.9371]])

rnn.weight_hh_l0
tensor([[ 0.7371,  0.4959,  1.1426],
        [-0.4023,  0.6238,  0.2884],
        [-0.3352, -0.1441, -0.0063]])

fc.weight
tensor([[-2.4486,  0.3788,  2.1328],
        [ 2.6678,  1.8949,  0.7092],
        [ 0.0934, -2.6197, -2.3509]])



### Discussion

In the `DishRNN` model, `hidden_size` refers to the number of features in the hidden state `h` of the RNN. It determines the dimensionality of the internal representation that the RNN uses to process the input sequence. A larger `hidden_size` allows the model to learn more complex patterns and relationships in the data, but also increases the number of parameters and computational cost.

This lab demonstrates a simple RNN model (`DishRNN`) to predict the next dish based on the current dish and weather conditions. The model successfully learns the transitions defined in the `generate_sequence` function, achieving 100% accuracy on the generated dataset.

The `DishRNN` takes an input of size 5 (3 for dish one-hot encoding + 2 for weather one-hot encoding) and outputs a prediction of size 3 (for the next dish). The `hidden_size` of 3 means that the internal state of the RNN has 3 features, which is sufficient for this particular problem to capture the simple state transitions.

### Conclusion

This lab provided a practical understanding of Recurrent Neural Networks (RNNs) by implementing a simple `DishRNN` model. We successfully built a model that can predict the next dish based on the current dish and weather conditions, achieving 100% accuracy on the generated dataset. The experiment highlighted the role of `hidden_size` in defining the internal state's complexity and its influence on the model's ability to capture sequential dependencies. The `DishRNN` effectively learned the specified state transitions, demonstrating the fundamental principles of sequence modeling with RNNs.